# 02-6. テイラー展開 — 動かして確かめる

📖 解説: [`../06_taylor_series.md`](../06_taylor_series.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. 素朴実装 vs 漸化式実装 (`exp` を題材に)
2. `sin` / `cos` の実装と検算
3. 【対話】次数を上げると本物に近づく (スライダー)
4. ⚠️ 収束半径 — `log` を壊して確かめる
5. 【対話】発散する瞬間を見る (スライダー)
6. 桁落ち — 数学的に正しいのに答えが合わない
7. JAX: `grad` の重ねがけで展開係数を自動取得
8. JAX: `jit` + 配列演算で一気に評価
9. softmax の max shift (実務での応用)

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (06_taylor_series.md)](../06_taylor_series.md)

In [ ]:
import math
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
import sympy as sp
import jax
import jax.numpy as jnp
import jax.scipy.special
from ipywidgets import interact, IntSlider, FloatSlider

## 1. 素朴実装 vs 漸化式実装

$$
e^x = \sum_{n=0}^{\infty} \frac{x^n}{n!}
$$

まず定義をそのままコードにします (素朴版)。次に、**前の項から次の項を作る**書き方 (漸化式版) を書きます。

$$
t_{i+1} = \frac{x^{i+1}}{(i+1)!} = \frac{x^i}{i!} \cdot \frac{x}{i+1} = t_i \cdot \frac{x}{i+1}
$$

掛け算 1 回で次の項が手に入るので、階乗も累乗も計算しなくて済みます。

In [ ]:
def exp_taylor_naive(x: float, n: int = 10) -> float:
    """exp(x) をマクローリン展開の第 n 項まで足して近似する (素朴版).

    定義 sum(x^i / i!) をそのままコードにした版。

    Args:
        x: 入力値 (単位なし、実数)
        n: 何項目まで足すか (n >= 0)

    Returns:
        exp(x) の近似値
    """
    return sum(x**i / math.factorial(i) for i in range(n + 1))


def exp_taylor(x: float, n: int = 10) -> float:
    """exp(x) をマクローリン展開の第 n 項まで足して近似する (漸化式版).

    t_{i+1} = t_i * x / (i+1) の関係を使い、各項を掛け算 1 回で作る。

    Args:
        x: 入力値 (単位なし、実数)
        n: 何項目まで足すか (n >= 0)

    Returns:
        exp(x) の近似値
    """
    total = 0.0
    term = 1.0  # t_0 = x^0 / 0! = 1
    for i in range(n + 1):
        total += term
        term *= x / (i + 1)  # 次の項へ更新
    return total


# 両者が一致することを確認
for n in (5, 10, 20, 30):
    a, b = exp_taylor_naive(1.0, n), exp_taylor(1.0, n)
    print(f'n={n:>3}  素朴={a:.15f}  漸化式={b:.15f}  一致={math.isclose(a, b)}')

print()
print(f'math.e     = {math.e:.15f}')

### 項数を増やすと誤差はどう減るか

誤差を**対数目盛**で見ると、テイラー展開の収束の速さが一目でわかります。

In [ ]:
ns = list(range(1, 21))                                  # 項数
errors = [abs(exp_taylor(1.0, n) - math.e) for n in ns]  # |近似 - 真値|

plt.figure(figsize=(8, 4))
plt.semilogy(ns, errors, 'o-')          # y 軸を対数目盛に
plt.axhline(2.2e-16, ls='--', c='r', label='float64 の限界 (約 1e-16)')
plt.xlabel('項数 n')
plt.ylabel('絶対誤差 |近似 - e|')
plt.title('exp(1) のテイラー近似: 項を増やすほど誤差は激減する')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print('n=18 あたりで float64 の精度限界に到達し、そこから先は改善しません。')

## 2. sin / cos の実装

$\sin$ と $\cos$ は 1 つ飛ばしの項なので、更新の比が変わります:

$$
t_{i+2} = t_i \cdot \frac{-x^2}{(i+1)(i+2)}
$$

分子の $-x^2$ が、**符号の交代**と **2 つ飛ばしの累乗**を同時に引き受けているのがポイントです。

In [ ]:
def sin_taylor(x: float, n_terms: int = 8) -> float:
    """sin(x) をマクローリン展開で近似する (漸化式版).

    Args:
        x: 角度 (単位: ラジアン)
        n_terms: 実際に足す項数 (次数ではないことに注意)

    Returns:
        sin(x) の近似値
    """
    total = 0.0
    term = x  # 第 1 項は x
    for i in range(n_terms):
        total += term
        # x^(2i+1) -> x^(2i+3) への更新 (符号反転込み)
        term *= -(x**2) / ((2 * i + 2) * (2 * i + 3))
    return total


def cos_taylor(x: float, n_terms: int = 8) -> float:
    """cos(x) をマクローリン展開で近似する (漸化式版).

    Args:
        x: 角度 (単位: ラジアン)
        n_terms: 実際に足す項数

    Returns:
        cos(x) の近似値
    """
    total = 0.0
    term = 1.0  # 第 1 項は 1
    for i in range(n_terms):
        total += term
        term *= -(x**2) / ((2 * i + 1) * (2 * i + 2))
    return total


print('x            sin近似            sin真値            cos近似            cos真値')
print('-' * 84)
for x in (1.0, math.pi / 6, math.pi / 4, math.pi / 3):
    print(f'{x:.5f}  {sin_taylor(x):>17.15f}  {math.sin(x):>17.15f}  '
          f'{cos_taylor(x):>17.15f}  {math.cos(x):>17.15f}')

### SymPy に答え合わせしてもらう

手で導いた展開式が合っているか、記号計算で確認できます。

In [ ]:
x_sym = sp.Symbol('x')

print('exp:', sp.series(sp.exp(x_sym), x_sym, 0, 6))
print('sin:', sp.series(sp.sin(x_sym), x_sym, 0, 8))
print('cos:', sp.series(sp.cos(x_sym), x_sym, 0, 8))
print('log:', sp.series(sp.log(x_sym), x_sym, 1, 4))  # log だけ中心が 1

print()
print('末尾の O(x**6) は「ランダウの記号」。ここから先は省略した、という意味。')

## 3. 【対話】次数を上げると本物に近づく

スライダーで項数を動かして、近似曲線が $\sin$ に貼りついていく様子を見てください。

**注目してほしいこと**: 項数を増やすと、まず $x=0$ の近くから一致し始めて、
だんだん**外側へ**一致する範囲が広がっていきます。テイラー展開が「中心の近くで正確」な近似だとわかります。

In [ ]:
def plot_sin_approx(n_terms: int = 1) -> None:
    """sin(x) とそのテイラー近似を重ねて描く.

    Args:
        n_terms: 足す項数
    """
    xs = np.linspace(-3 * np.pi, 3 * np.pi, 400)          # shape: (400,)
    ys_true = np.sin(xs)                                   # shape: (400,)
    ys_approx = np.array([sin_taylor(x, n_terms) for x in xs])  # shape: (400,)

    plt.figure(figsize=(9, 5))
    plt.plot(xs, ys_true, label='sin(x) (本物)', lw=2)
    plt.plot(xs, ys_approx, '--', label=f'テイラー近似 ({n_terms} 項 = {2*n_terms-1} 次)', lw=2)
    plt.ylim(-2, 2)          # 発散して画面外に飛ぶので範囲を固定
    plt.axhline(0, c='k', lw=0.5)
    plt.axvline(0, c='k', lw=0.5, label='展開の中心 x=0')
    plt.title(f'sin(x) のマクローリン近似: {n_terms} 項')
    plt.legend(loc='upper right')
    plt.grid(alpha=0.3)
    plt.show()


interact(plot_sin_approx, n_terms=IntSlider(min=1, max=12, step=1, value=1))

## 4. ⚠️ 収束半径 — `log` を壊して確かめる

ここがこのノートの核心です。

$\exp$ では「項を増やすほど正確になる」でした。では $\log$ でも同じことが起きるでしょうか。

$$
\log x = \sum_{n=1}^{\infty} \frac{(-1)^{n-1}(x-1)^n}{n}
$$

**予想してから実行してください。** $x=3$ で項数を 10 → 30 → 100 と増やしたら、答えはどうなると思いますか?

In [ ]:
def log_taylor(x: float, n: int = 10) -> float:
    """log(x) を x=1 中心のテイラー展開で近似する.

    Args:
        x: 入力値 (x > 0)
        n: 何項まで足すか (n >= 1)

    Returns:
        log(x) の近似値 (収束するのは 0 < x <= 2 のみ)
    """
    total = 0.0
    term = -1.0
    for i in range(1, n + 1):
        term *= -(x - 1)      # (-1)^(i-1) * (x-1)^i を作る
        total += term / i
    return total


print(f'{"x":>5} {"真の値":>16} {"n=10":>18} {"n=30":>18} {"n=100":>18}')
print('-' * 80)
for x in (1.1, 1.5, 2.0, 2.5, 3.0):
    print(f'{x:>5} {math.log(x):>16.10f} '
          f'{log_taylor(x, 10):>18.10g} {log_taylor(x, 30):>18.10g} {log_taylor(x, 100):>18.10g}')

### 結果の読み方

$x = 3$ の行を見てください。真の値は $1.0986\ldots$、つまり **1 くらい**のはずです。

なのに近似値は $-64$ → $-2359$ 万 → $-8.4 \times 10^{27}$。

**項を増やすほど、答えから猛烈な勢いで遠ざかっています。**

これはバグではありません。第 $n$ 項の分子 $(x-1)^n = 2^n$ が指数関数的に爆発する一方、
分母の $n$ は 1 次でしか増えないからです。**指数関数は多項式に必ず勝ちます。**

この境界 (中心からどこまで離れて良いか) を **収束半径 (radius of convergence)** と呼びます。
$\log$ の中心 $a=1$ での収束半径は $R = 1$、つまり $0 < x < 2$ でしか使えません。

In [ ]:
# 誤差の推移をグラフで (対数目盛)
ns = list(range(1, 61))

plt.figure(figsize=(9, 5))
for x, style in [(1.1, 'o-'), (1.5, 's-'), (2.0, '^-'), (2.5, 'v--'), (3.0, 'x--')]:
    errs = [abs(log_taylor(x, n) - math.log(x)) for n in ns]
    plt.semilogy(ns, errs, style, markersize=3, label=f'x = {x}')

plt.xlabel('項数 n')
plt.ylabel('絶対誤差 (対数目盛)')
plt.title('log のテイラー近似: 収束半径の内と外で運命が分かれる')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print('下がっていく線 = 収束 (x < 2)  /  上がっていく線 = 発散 (x > 2)')
print('x = 2.0 はぎりぎり境界。収束はするが絶望的に遅い。')

## 5. 【対話】発散する瞬間を見る

$x$ をスライダーで動かして、**$x = 2$ を超えた瞬間に何が起きるか**を確かめてください。

In [ ]:
def plot_log_convergence(x: float = 1.5, n_max: int = 40) -> None:
    """log の部分和が項数とともにどう動くかを描く.

    Args:
        x: 評価する点 (x > 0)
        n_max: 最大項数
    """
    ns = list(range(1, n_max + 1))
    partial_sums = [log_taylor(x, n) for n in ns]
    true_value = math.log(x)

    plt.figure(figsize=(9, 5))
    plt.plot(ns, partial_sums, 'o-', markersize=4, label='部分和 (近似値)')
    plt.axhline(true_value, c='r', ls='--', lw=2, label=f'真の値 log({x}) = {true_value:.6f}')
    plt.xlabel('項数 n')
    plt.ylabel('近似値')

    inside = abs(x - 1) < 1
    verdict = '収束圏内 ✅' if inside else '収束圏外 ❌ (発散)'
    plt.title(f'x = {x}  |x - 1| = {abs(x-1):.2f}  →  {verdict}')

    # 発散すると縦軸が潰れるので、真値まわりに範囲を固定
    plt.ylim(true_value - 3, true_value + 3)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    print(f'n={n_max} での近似値: {partial_sums[-1]:.6g}   (真の値: {true_value:.6g})')


interact(
    plot_log_convergence,
    x=FloatSlider(min=0.3, max=3.0, step=0.1, value=1.5),
    n_max=IntSlider(min=5, max=60, step=5, value=40),
)

## ⚠️ よくあるエラー — 数学的に正しいのに答えが合わない

$\exp$ の収束半径は $\infty$ です。つまり**どんな $x$ でも数学的には必ず収束します**。

では $e^{-20}$ を $n=50$ で計算してみましょう。真の値は $2.06 \times 10^{-9}$ です。

さて、いくつが返るでしょうか。

In [ ]:
print(f'{"x":>5} {"テイラー (n=50)":>24} {"真の値":>16}')
print('-' * 50)
for x in (-1, -10, -20, -30):
    print(f'{x:>5} {exp_taylor(x, 50):>24.10g} {math.exp(x):>16.8g}')

print()
print('x = -20 で答えが 9 桁ずれています。収束半径の問題ではありません。')

### 犯人は「途中の項が巨大になること」

$x = -20$ のとき、途中の項は最大で $\dfrac{20^{20}}{20!} \approx 4.3 \times 10^7$ にもなります。

$4 \times 10^7$ 規模の正の項と負の項が打ち消し合って、最終的に $2 \times 10^{-9}$ という答えを作る。
ところが `float64` の有効数字は約 16 桁しかないので、**答えに必要な桁が計算の途中で消し飛びます**。

これを **桁落ち (catastrophic cancellation)** と呼びます。

In [ ]:
# 途中の項がどれだけ大きくなるか見てみる
x = -20.0
term = 1.0
terms = []
for i in range(51):
    terms.append(term)
    term *= x / (i + 1)

max_term = max(abs(t) for t in terms)
print(f'最大の項の絶対値: {max_term:.4g}')
print(f'最終的な答え    : {math.exp(-20):.4g}')
print(f'比率            : {max_term / math.exp(-20):.4g}  ← この比率のぶんだけ桁が失われる')
print()

# 対処: 数学的に等価な式に書き換える
print('❌ そのまま        :', exp_taylor(-20, 50))
print('✅ 1/exp_taylor(20):', 1.0 / exp_taylor(20, 100))
print('   真の値          :', math.exp(-20))

### 浮動小数点のもう 1 つの罠: `==` で比較しない

In [ ]:
approx = exp_taylor(1, 30)

print('近似値 :', approx)
print('math.e :', math.e)
print()
print('approx == math.e            →', approx == math.e)          # False ← 罠
print('math.isclose(approx, math.e) →', math.isclose(approx, math.e))  # True ← 正しい
print()
print(f'差: {abs(approx - math.e):.3e}  (float64 の最後の 1 ビット分)')

## 6. JAX: `grad` の重ねがけで展開係数を自動取得

テイラー展開の一般形をもう一度:

$$
f(x) = \sum_{n=0}^{\infty} \frac{f^{(n)}(a)}{n!}(x-a)^n
$$

係数を作るには**何階でも微分した値**が必要です。`jax.grad` は「微分された関数」を返すので、
**重ねがけすれば何階でも取れます**。公式を一切与えずに係数が出てくるのが見どころです。

In [ ]:
def nth_derivative(f: Callable[[float], float], n: int) -> Callable[[float], float]:
    """関数 f の n 階導関数を返す.

    jax.grad は「微分された関数」を返すので、n 回適用すれば n 階微分になる。

    Args:
        f: 微分対象のスカラー関数
        n: 微分の階数 (n >= 0)

    Returns:
        n 階導関数
    """
    for _ in range(n):
        f = jax.grad(f)
    return f


def taylor_coeffs(f: Callable[[float], float], a: float, n_terms: int) -> list[float]:
    """f の a を中心とするテイラー係数 f^(n)(a) / n! を求める.

    Args:
        f: 対象の関数
        a: 展開の中心
        n_terms: 求める係数の個数

    Returns:
        係数のリスト (長さ n_terms)
    """
    return [
        float(nth_derivative(f, n)(a)) / math.factorial(n)
        for n in range(n_terms)
    ]


print('sin の係数 (0 中心):')
for i, c in enumerate(taylor_coeffs(jnp.sin, 0.0, 8)):
    note = ''
    if i == 1:
        note = '  ← x の係数 1'
    elif i == 3:
        note = '  ← x^3 の係数 -1/6'
    elif i == 5:
        note = '  ← x^5 の係数 1/120'
    print(f'  x^{i}: {c:>22.18f}{note}')

print()
print('公式を 1 つも書いていないのに、grad を重ねるだけで係数が出ました。')
print('偶数番目が 0 なのは sin が奇関数だから。')

In [ ]:
# exp なら全係数が 1/n!
print('exp の係数 (0 中心):')
for i, c in enumerate(taylor_coeffs(jnp.exp, 0.0, 6)):
    print(f'  x^{i}: {c:.10f}   (1/{i}! = {1/math.factorial(i):.10f})')

## 7. JAX: `jit` + 配列演算で一気に評価

近似曲線を描くには、たくさんの $x$ で近似値を計算します。
Python の `for` ループを配列演算に置き換えると、`jit` でコンパイルできて高速になります。

> ⚠️ `n_terms` のようにループ回数を決める引数は `jit` の中では**静的 (static)** でなければなりません。
> `jnp.arange(n_terms)` の形にして配列演算に落とし込むことで、Python の `for` を消しています。

In [ ]:
@jax.jit
def sin_taylor_jax(x: jnp.ndarray, n_terms: int = 8) -> jnp.ndarray:
    """sin(x) のマクローリン近似 (JAX 版, jit 対応).

    Args:
        x: 角度 (単位: ラジアン), shape: (N,)
        n_terms: 足す項数 (静的な値として扱われる)

    Returns:
        sin(x) の近似値, shape: (N,)
    """
    i = jnp.arange(n_terms)                                        # shape: (n_terms,)
    powers = 2 * i + 1                                             # shape: (n_terms,)
    signs = (-1.0) ** i                                            # shape: (n_terms,)
    factorials = jnp.exp(jax.scipy.special.gammaln(powers + 1.0))  # (2i+1)!
    coeffs = signs / factorials                                    # shape: (n_terms,)
    # x[..., None] で shape (N, 1) にして (n_terms,) とブロードキャスト
    return jnp.sum(coeffs * x[..., None] ** powers, axis=-1)       # shape: (N,)


xs = jnp.linspace(-jnp.pi, jnp.pi, 5)     # shape: (5,)
print('テイラー:', sin_taylor_jax(xs))
print('本物    :', jnp.sin(xs))
print()
print('JAX のデフォルトは float32 なので、NumPy (float64) より精度が低く出ます。')
print('これは仕様です。詳しくは 07_jax/01_basics.md を参照。')

## 8. softmax の max shift — 実務での応用

第 6 節の桁落ちと**まったく同じ発想**が、ML で最も有名な形で現れるのが softmax です。

$$
\mathrm{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}
= \frac{e^{z_i - \max_j z_j}}{\sum_k e^{z_k - \max_j z_j}}
$$

分子・分母に同じ $e^{-\max z}$ を掛けただけなので**数学的には完全に同じ式**。
でも指数の中身が必ず $0$ 以下になるので `inf` が出ません。

In [ ]:
def softmax_naive(z: np.ndarray) -> np.ndarray:
    """素朴な softmax (大きな入力で overflow する).

    Args:
        z: ロジット, shape: (N,)

    Returns:
        確率ベクトル, shape: (N,)
    """
    e = np.exp(z)
    return e / e.sum()


def softmax_stable(z: np.ndarray) -> np.ndarray:
    """数値的に安定な softmax (max shift 版).

    Args:
        z: ロジット, shape: (N,)

    Returns:
        確率ベクトル, shape: (N,)
    """
    e = np.exp(z - z.max())  # 指数の中身を必ず 0 以下にする
    return e / e.sum()


z = np.array([1000.0, 1001.0, 1002.0])   # shape: (3,)

with np.errstate(over='ignore', invalid='ignore'):
    print('素朴版  :', softmax_naive(z), '  ← overflow して nan')
print('安定版  :', softmax_stable(z))
print('jax.nn  :', jax.nn.softmax(jnp.array(z)), '  ← ライブラリは当然安定版')

## まとめ

- **テイラー展開** = 関数を $\sum \frac{f^{(n)}(a)}{n!}(x-a)^n$ という多項式に置き換える技
- **実装のコツ** = 階乗を毎回計算せず、$t_{i+1} = t_i \cdot (\text{比})$ の漸化式で回す
- **収束半径** = $\exp, \sin, \cos$ は $R=\infty$。$\log$ は $R=1$ で、範囲外では項を増やすほど悪化する
- **桁落ち** = 数学的に正しくても、途中の項が巨大だと有効数字が消える → 等価な式に書き換える
- **`==` 禁止** = 浮動小数点の比較は `math.isclose` / `np.isclose` で
- **JAX** = `jax.grad` の重ねがけで高階微分 → 展開係数が公式なしで出る
- **ML への接続** = softmax の max shift は、桁落ち対策そのもの

一番の持ち帰り:

> **近似には必ず有効範囲がある。範囲を確認せずに使うと、努力するほど間違いが大きくなる。**

## 微積分章 完了 🎉

これで 02_calculus 章は卒業です。次は確率・統計へ。

→ 次の章: [`../../03_probability_statistics/README.md`](../../03_probability_statistics/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`05_gradient_jacobian.ipynb`](05_gradient_jacobian.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../03_probability_statistics/README.md`](../../03_probability_statistics/README.md) |